# Checking for dropped camera frames

This notebook verifies that every camera trigger issued by the Behavior board during a session produced a logged
frame. The readers and loaders are shared with `inspect_session.ipynb` through `session_io.py`.

In [1]:
from pathlib import Path

import pandas as pd
import swc.aeon.io.api as aeon

from session_io import declare_experiment, find_device_schema, frame_count_check, load_camera

## Dataset location

`root` is a single session directory. It can also be any parent directory (for example `sub-Plimbo`) to stitch
together data from several sessions.

Harp devices need the device schema to decode registers. The logging specification requires it to be saved as
`<Device>/<Device>.yml`. If it is not present, `find_device_schema` downloads the schema for the Harp Behavior
board from [harp-tech/device.behavior](https://github.com/harp-tech/device.behavior).

In [2]:
root = Path("../temp_data/sub-Plimbo/ses-001_date-2026-10-15T15-18-01")
device_yml = find_device_schema(root)

## Declaring the experiment

`declare_experiment` maps each device directory to the readers for its data streams. The Behavior board is a
multi-channel Harp device, so `harp_device` creates one reader per register in the schema, accessed as
`experiment.BehaviorBoard.<Register>`. The two cameras are declared with `camera_video`, a reader for the fixed
`<Device>/<Device>.csv` + `<Device>.avi` pair written by this workflow (see `session_io.py`); it produces the same
columns as `open_rigs.io.reader.RawVideo`, so the data frames work with the aeon video helpers.

In [3]:
experiment = declare_experiment(device_yml)
board = experiment.BehaviorBoard

print(list(experiment.keys()))

['BehaviorBoard', 'SubjectCamera', 'TrackCamera']


## Loading camera frames

`load_camera` returns one row per logged frame, indexed by the frame's Harp timestamp.

In [4]:
subject_video = load_camera(root, experiment.SubjectCamera)
track_video = load_camera(root, experiment.TrackCamera)
subject_video.head()

,_frame,_path,_epoch
time,,,
1904-02-18 23:24:01.195488+00:00,0,..\temp_data\sub-Plimbo\ses-001_date-2026-10-1...,ses-001_date-2026-10-15T15-18-01
1904-02-18 23:24:02.349600+00:00,1,..\temp_data\sub-Plimbo\ses-001_date-2026-10-1...,ses-001_date-2026-10-15T15-18-01
1904-02-18 23:24:02.356224+00:00,2,..\temp_data\sub-Plimbo\ses-001_date-2026-10-1...,ses-001_date-2026-10-15T15-18-01
1904-02-18 23:24:02.362912+00:00,3,..\temp_data\sub-Plimbo\ses-001_date-2026-10-1...,ses-001_date-2026-10-15T15-18-01
1904-02-18 23:24:02.369568+00:00,4,..\temp_data\sub-Plimbo\ses-001_date-2026-10-1...,ses-001_date-2026-10-15T15-18-01


## Frames versus trigger events

Each camera is hardware-triggered by the board (`Camera0` drives the subject camera, `Camera1` the track camera)
and every trigger pulse is logged as a `Camera0Frame` / `Camera1Frame` event. `frame_count_check` compares the
number of frames logged after `StartCameras` with the number of trigger events. Frames before `StartCameras` are
not trigger-driven (the register-dump read reply logged at startup) and are excluded; triggers issued after the
last logged frame are expected, since the workflow stops before their frame arrives, and are reported separately.
A camera passes when the two counts are equal.

In [5]:
cameras_started = aeon.load(root, board.StartCameras).index.min()

report = pd.DataFrame(
    {
        "SubjectCamera": frame_count_check(subject_video, aeon.load(root, board.Camera0Frame), cameras_started),
        "TrackCamera": frame_count_check(track_video, aeon.load(root, board.Camera1Frame), cameras_started),
    }
)
report

,SubjectCamera,TrackCamera
frames after StartCameras,1900,1900
triggers,1901,1901
triggers after last frame,1,1
dropped frames,0,0
pass,True,True


In [6]:
for name, result in report.items():
    print(f"{name}: {'PASS' if result['pass'] else 'FAIL'} ({result['dropped frames']} dropped frames)")

SubjectCamera: PASS (0 dropped frames)
TrackCamera: PASS (0 dropped frames)
